<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# WebRTC VAD and Silero VAD

**[WebRTC VAD and Silero VAD](https://learning.nextia-ai.com/courses/speech/m03/webrtc-vad-and-silero-vad/)** · Speech AI: Transcribe, Understand and Speak · Module 3, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** run two real voice activity detectors, WebRTC VAD and Silero VAD, score them against the true turns with the energy VAD of the last lesson, see what each one calls speech in the noise set, and choose a Silero threshold for the phone line.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account, no key. Silero VAD comes inside faster-whisper (a 1.2 MB file); no recogniser is downloaded. |
| **You should know** | The energy VAD, missed speech and false speech, from [Speech or silence?](https://learning.nextia-ai.com/courses/speech/m03/speech-or-silence/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1, webrtcvad-wheels 2.0.14.post1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m03/webrtc-vad-and-silero-vad/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M03-L02-webrtc-vad-and-silero-vad/webrtc-vad-and-silero-vad-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, faster-whisper 1.2.1 (it brings Silero VAD) and webrtcvad-wheels 2.0.14.post1 (WebRTC VAD), and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time. This lesson downloads no recogniser.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2 faster-whisper==1.2.1 webrtcvad-wheels==2.0.14.post1
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ["numpy", "scipy", "soundfile", "faster-whisper", "webrtcvad-wheels"]:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 8 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/calls/calls.jsonl": "22fe4e147bf62b4ff6c3b6b4a1cc559b048e12a0c2b3c8098eb72982f2d9e695",
    "data/calls/phone/C-0022.wav": "f1958f7b460c5f040dfd0f84a31cb83677ee66b25b514df21a086d9d16db5a51",
    "data/calls/phone/C-0098.wav": "58d8793dc53dcedc78b0d1f8f9616ea38044d8f76a321c76ab007cbaaa39bfca",
    "data/calls/phone/C-0204.wav": "2fc187d2960de73f32beb554c90781e0a2853520dfce12582d47de0e5f6911e5",
    "data/calls/phone/C-0093.wav": "68125fe62023b1ec52064e488a45ffdcce48fa5c602ece56c2fa16f2a9a2e424",
    "data/calls/phone/C-0181.wav": "6bce67f941673d66ec1a2cf271c4c7872721a7936dbb841ac927bc856ab96726",
    "data/calls/phone/C-0003.wav": "e03c761cb765170c39bcc4d676e3f4f7144f510c38ae476182d4cb5432792761",
    "data/noise/music_on_hold.wav": "ecdc8ca83a89079c2acd2e4d9d93d47fb725650c25643de67f9332645a1771fd",
    "data/noise/background_talk.wav": "232f4c029b07247ca4207d8752e1cdb7784eb35222218176d12470ce23355ab0",
    "data/noise/street.wav": "8474ae365d3231942d439734fc2ac0b8bf499be7382b382de1eaf4d888c299fb",
    "data/noise/line.wav": "560e52c3c810a50803e6e5af0eeaaa7febff384f46519033d9739edaa5c170e4",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The tools from the last lesson

The next cell has the energy VAD and the scoring function from [Speech or silence?](https://learning.nextia-ai.com/courses/speech/m03/speech-or-silence/). Run it. It prints nothing.

In [ ]:
FRAME = 0.03  # 30 ms frames

def frame_db(x, sr):
    """The loudness of each 30 ms frame, in dBFS."""
    n = int(sr * FRAME)
    frames = x[: len(x) // n * n].reshape(-1, n)
    return 20 * np.log10(np.sqrt(np.mean(frames ** 2, axis=1)) + 1e-9)

def regions(flags, hangover=0.3, min_speech=0.1):
    """Speech frames -> (start, end) times. A pause shorter than `hangover` seconds stays inside the speech."""
    out, start, quiet = [], None, 0
    keep = int(round(hangover / FRAME))
    for i, f in enumerate(flags):
        if f:
            start = i if start is None else start
            quiet = 0
        elif start is not None:
            quiet += 1
            if quiet > keep:
                end = i - quiet + 1
                if (end - start) * FRAME >= min_speech:
                    out.append((start * FRAME, end * FRAME))
                start, quiet = None, 0
    if start is not None:
        out.append((start * FRAME, (len(flags) - quiet) * FRAME))
    return out

def energy_vad(x, sr, threshold_db=-40.0, hangover=0.3):
    """The baseline: a frame is speech when it is louder than threshold_db."""
    return regions(frame_db(x, sr) > threshold_db, hangover)

def frame_scores(pred, truth, seconds, step=0.01):
    """missed = share of true speech not found; false = share of non-speech called speech."""
    n = int(seconds / step) + 1
    def mask(spans):
        m = np.zeros(n, bool)
        for s, e in spans:
            m[int(s / step):int(e / step)] = True
        return m
    p, t = mask(pred), mask(truth)
    return {"missed": float((t & ~p).sum() / max(t.sum(), 1)), "false": float((p & ~t).sum() / max((~t).sum(), 1))}

## 2. Two detectors that look at the shape of the sound

**WebRTC VAD** comes from the WebRTC project, the code that browsers use for voice calls. For each 30 ms frame, it looks at the energy in a few frequency bands and gives a yes or no. Its **mode** (0 to 3) says how strict it is: mode 3 says "speech" least often.

**Silero VAD** is a small neural network (a 1.2 MB file) that learned from many hours of speech and noise. For each chunk of 32 ms, it gives a **probability** of speech between 0 and 1. A **threshold** turns the probability into yes or no. faster-whisper ships a copy of it, so you already have it.

Run the next two cells. They define `webrtc_vad` and `silero_vad`, with the settings of the course project.

In [ ]:
import webrtcvad

def webrtc_vad(x, sr, mode=2, hangover=0.3):
    """WebRTC VAD: a yes/no answer for each 30 ms frame. mode 0 (lenient) to 3 (strict). sr: 8000 or 16000."""
    vad = webrtcvad.Vad(mode)
    n = int(sr * FRAME)
    pcm = (np.clip(x, -1, 1) * 32767).astype("<i2")          # 16-bit numbers, as the detector expects
    flags = [vad.is_speech(pcm[i:i + n].tobytes(), sr) for i in range(0, len(pcm) - n + 1, n)]
    return regions(np.array(flags), hangover)

In [ ]:
from faster_whisper.vad import VadOptions, get_speech_timestamps, get_vad_model

def silero_probs(x16):
    """Silero's speech probability (0 to 1) for each chunk of 512 samples: 32 ms at 16 kHz."""
    return get_vad_model()(np.pad(x16, (0, 512 - len(x16) % 512)))

def silero_vad(x16, threshold=0.5):
    """Silero VAD on 16 kHz audio: speech where the probability is above the threshold."""
    opts = VadOptions(threshold=threshold, min_silence_duration_ms=300, speech_pad_ms=100)
    return [(t["start"] / 16000, t["end"] / 16000) for t in get_speech_timestamps(x16, opts, sampling_rate=16000)]

## 3. A tiny example: Silero's probabilities

Call C-0022 is the call with music on hold from the last lesson. From 21.9 to 24.9 seconds, the agent says "Thank you. Let me check that for you. Please hold." The next cell prints Silero's probability for 10 chunks in the middle of that sentence: about a third of a second.

> **Predict.** The agent is speaking the whole time. Will all 10 probabilities be above 0.5?

In [ ]:
calls = load_calls()
call = calls["C-0022"]
x8, _ = read_audio("data/calls/phone/C-0022.wav")            # 8 kHz, as the phone line recorded it
x16, _ = read_audio("data/calls/phone/C-0022.wav", sr=16000)  # the same audio at 16 kHz, for Silero
probs = silero_probs(x16)
first = int(22.76 * 16000 / 512)                               # the chunk that starts at 22.75 s
for i in range(first, first + 10):
    print(f"{i * 512 / 16000:6.2f} s   probability {probs[i]:.2f}   speech at 0.5: {probs[i] > 0.5}")

> **Check.** You should see these 10 probabilities: [0.48, 0.51, 0.5, 0.49, 0.45, 0.34, 0.22, 0.26, 0.5, 0.47]. The agent speaks in every chunk, but the probabilities stay close to 0.5, with a dip in a short pause between two words. At 0.5, about half the chunks are "not speech". Change 0.5 to 0.3 in your head: which chunks change their answer? Silero is unsure about this voice on the phone line, and that is the main fact of this lesson.

## 4. Three detectors on one call

In [ ]:
truth = [(s, e) for s, e, _ in true_turns(call)]
found = {"energy": energy_vad(x8, 8000, -40), "webrtc": webrtc_vad(x8, 8000, mode=2), "silero": silero_vad(x16, 0.5)}
hold = [t for t in call["turns"] if t.get("hold")][0]
hold_start, hold_end = hold["start"] - hold["silence_before"], hold["start"]
for name, pred in found.items():
    s = frame_scores(pred, truth, call["duration"])
    in_hold = sum(max(0, min(e, hold_end) - max(a, hold_start)) for a, e in pred)
    print(f"{name:7} {len(pred):3} regions   missed {s['missed']:.3f}   false {s['false']:.3f}   "
          f"music called speech: {in_hold:4.1f} of {hold_end - hold_start:.1f} s")

> **Check.** Energy: missed 0.19, false 0.626, and 17.1 of the 17.7 seconds of music called speech. WebRTC VAD: missed 0.106, false 0.638, music 17.2 s. Silero VAD: missed 0.411, false 0.016, music 0 s.

Silero is the only one that hears that music is not a voice. But look at its missed speech: it is the highest of the three. The next cell draws the regions.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 2.8))
rows = [("true speech", truth, "green")] + [(n, p, c) for (n, p), c in zip(found.items(), ["grey", "tab:orange", "tab:blue"])]
for k, (name, spans, colour) in enumerate(rows):
    for s, e in spans:
        ax.barh(len(rows) - 1 - k, e - s, left=s, height=0.6, color=colour)
ax.axvspan(hold_start, hold_end, color="gold", alpha=0.2, label="music on hold")
ax.set_yticks(range(len(rows))); ax.set_yticklabels([r[0] for r in rows][::-1])
ax.set_xlabel("seconds"); ax.legend(loc="upper right"); ax.set_title("C-0022: where each detector finds speech")
plt.show()

## 5. The noise set

> **Predict.** Background talk is people speaking. Which detector will call it speech?

In [ ]:
for name in ["music_on_hold", "background_talk", "street", "line"]:
    n8, _ = read_audio(f"data/noise/{name}.wav", sr=8000)
    n16, _ = read_audio(f"data/noise/{name}.wav", sr=16000)
    share = lambda spans: sum(e - s for s, e in spans) / 20
    print(f"{name:16} energy {share(energy_vad(n8, 8000, -40)):.2f}   webrtc {share(webrtc_vad(n8, 8000)):.2f}   "
          f"silero {share(silero_vad(n16)):.2f}")

> **Check.** Music on hold: energy 1.0, WebRTC 1.0, Silero 0.03. Background talk: 1.0, 1.0, 1.0. Street: 1.0, 0.56, 0.0. Line hiss: 0.0, 0.01, 0.0.

Every detector calls background talk speech, because it **is** speech. A voice detector cannot tell whose voice it hears. (Here the noise is read at 8 kHz without the phone line's filter, so the numbers differ a little from the lesson's table.)

## 6. Speed

In [ ]:
import time
minutes = call["duration"] / 60
for name, run in [("energy", lambda: energy_vad(x8, 8000, -40)), ("webrtc", lambda: webrtc_vad(x8, 8000)),
                  ("silero", lambda: silero_vad(x16))]:
    run()                                    # the first run loads the model: do not time it
    t0 = time.perf_counter(); run(); ms = 1000 * (time.perf_counter() - t0)
    print(f"{name:7} {ms / minutes:7.1f} ms of compute per minute of audio")

> **Check.** Your numbers will differ, but the order will not: energy is the fastest, WebRTC VAD is next, and Silero is about 20 to 500 times slower than the others. Even Silero needs well under a second of compute for a minute of audio.

## 7. Failure case: the wrong sample rate

Silero, as faster-whisper calls it, expects 16,000 samples a second. A phone call has 8,000. The next cell gives it the 8 kHz audio by mistake.

In [ ]:
wrong = silero_vad(x8, 0.5)   # 8 kHz samples, but the function treats them as 16 kHz
s = frame_scores(wrong, truth, call["duration"])
print("last region ends at", round(wrong[-1][1], 1), "s; the call lasts", call["duration"], "s")
print(f"missed {s['missed']:.3f}, false {s['false']:.3f}")

> **Check.** The last region ends at 24.2 s, but the call lasts 70.394 s. The function counts 16,000 samples as one second, so every time is half the real time, and missed speech jumps to 0.859. No error message warns you. **Fix:** resample to 16 kHz first (`read_audio(path, sr=16000)`), as the project's `silero()` does.

## 8. Your turn: choose a threshold for the phone line

> **Your turn.** For each threshold in `[0.2, 0.3, 0.4, 0.5, 0.6, 0.7]`, run `silero_vad` on the six calls in `task_calls` (16 kHz, both channels mixed), score each call against its true turns, and keep the **mean missed** and **mean false** speech in a dictionary `results` (threshold → (missed, false)). Then put in `chosen` the **highest** threshold whose mean missed speech is at most 0.10. Missed speech is lost for the recogniser, so Larkfield allows at most 10% of it.

In [ ]:
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
task_calls = ["C-0022", "C-0098", "C-0204", "C-0093", "C-0181", "C-0003"]
audio16 = {c: read_audio(f"data/calls/phone/{c}.wav", sr=16000)[0] for c in task_calls}
results = {}
chosen = ...

In [ ]:
expect('chosen', chosen, 0.3)

In [ ]:
expect("its mean missed speech", None if chosen is ... else round(results[chosen][0], 3), 0.077, tolerance=0.002)

> **Check.** At the default 0.5, these six calls lose 0.281 of their speech (false speech 0.082). At 0.3, they lose 0.077, and false speech rises to 0.208. A lower threshold keeps more speech and lets in more noise. The next lesson measures what that trade does to the recogniser's errors and time.

## Next

You compared three detectors on one call and on the noise set, saw a silent mistake with the sample rate, and chose a threshold. The [lesson page](https://learning.nextia-ai.com/courses/speech/m03/webrtc-vad-and-silero-vad/) has the measurements on all 50 validation calls and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Why VAD makes recognition better](https://learning.nextia-ai.com/courses/speech/m03/why-vad-makes-recognition-better/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m03/webrtc-vad-and-silero-vad/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.